# Nemotron Trace Generator v1 — 2026-04-29

Purpose:
- read `train.csv`
- classify each puzzle into a category
- generate deterministic boxed-answer traces for a first set of categories
- export an SFT-ready corpus for later training

Scope for v1:
- numeral
- gravity
- unit_conversion

This notebook does **not** train the model and does **not** package a submission.


In [ ]:
import json
import math
import os
import re
from pathlib import Path

import polars as pl
import pandas as pd

INPUT_CANDIDATES = [
    '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv',
    '/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv',
]

TRAIN_PATH = None
for p in INPUT_CANDIDATES:
    if os.path.exists(p):
        TRAIN_PATH = p
        break

if TRAIN_PATH is None:
    raise FileNotFoundError(f'Could not find train.csv in {INPUT_CANDIDATES}')

OUTPUT_DIR = Path('/kaggle/working')
JSONL_PATH = OUTPUT_DIR / 'train_traces_v1.jsonl'
PREVIEW_PATH = OUTPUT_DIR / 'train_traces_v1_preview.csv'
STATS_PATH = OUTPUT_DIR / 'train_traces_v1_stats.csv'
SKIPPED_PATH = OUTPUT_DIR / 'train_traces_v1_skipped.csv'

train = pl.read_csv(TRAIN_PATH)
print('TRAIN_PATH:', TRAIN_PATH)
print('Train shape:', train.shape)
print(train.head())


In [ ]:
def classify_problem(prompt: str) -> str:
    p = prompt.lower()
    if 'roman' in p or 'numeral system' in p or 'xliv' in p or 'lxv' in p:
        return 'numeral'
    if 'd = 0.5*g*t^2' in prompt or 'gravitational constant' in p or 'falling distance' in p:
        return 'gravity'
    if 'unit conversion' in p or ('convert the following measurement' in p) or ('convert the following value' in p) or ('becomes' in p and re.search(r'\b(?:cm|mm|m|km|g|kg|mg|l|ml|ft|in|yd|mi)\b', p)):
        return 'unit_conversion'
    return 'other'

def format_answer_like_ground_truth(value: float, answer: str) -> str:
    ans = str(answer).strip()
    if re.fullmatch(r'-?\d+', ans):
        return str(int(round(value)))
    if re.fullmatch(r'-?\d+\.\d+', ans):
        decimals = len(ans.split('.')[-1])
        return f'{value:.{decimals}f}'
    return str(value)

def to_roman(n: int) -> str:
    vals = [
        (100, 'C'), (90, 'XC'), (50, 'L'), (40, 'XL'),
        (10, 'X'), (9, 'IX'), (5, 'V'), (4, 'IV'), (1, 'I')
    ]
    out = []
    for v, sym in vals:
        while n >= v:
            out.append(sym)
            n -= v
    return ''.join(out)


In [ ]:
def build_numeral_trace(row):
    prompt = row['prompt']
    answer = str(row['answer']).strip()

    match = re.search(r'write the number\s+(\d+)\s+in the wonderland numeral system', prompt, flags=re.IGNORECASE)
    if not match:
        return None
    target = int(match.group(1))

    hundreds = target // 100
    tens_ones = target % 100
    roman = to_roman(target)

    trace = (
        f'Step 1: The examples match standard Roman numerals.\n'
        f'Step 2: Convert the target number {target} into Roman numerals.\n'
        f'Step 3: {target} = {hundreds} hundreds and {tens_ones} remainder.\n'
        f'Step 4: Roman numeral form is {roman}.\n'
        f'Final answer: \boxed{{{answer}}}'
    )
    return trace

def parse_gravity_examples(prompt):
    examples = re.findall(r'For t =\s*([0-9]+(?:\.[0-9]+)?)s,\s*distance =\s*([0-9]+(?:\.[0-9]+)?)', prompt)
    query = re.search(r'determine the falling distance for t =\s*([0-9]+(?:\.[0-9]+)?)s', prompt, flags=re.IGNORECASE)
    if not examples or not query:
        return None
    ex = [(float(t), float(d)) for t, d in examples]
    target_t = float(query.group(1))
    return ex, target_t

def build_gravity_trace(row):
    prompt = row['prompt']
    answer = str(row['answer']).strip()
    parsed = parse_gravity_examples(prompt)
    if parsed is None:
        return None
    ex, target_t = parsed

    g_estimates = []
    for t, d in ex:
        if t != 0:
            g_estimates.append((2.0 * d) / (t * t))
    if not g_estimates:
        return None
    g = sum(g_estimates) / len(g_estimates)
    pred = 0.5 * g * target_t * target_t
    formatted_pred = format_answer_like_ground_truth(pred, answer)

    trace = (
        f'Step 1: Use d = 0.5 * g * t^2.\n'
        f'Step 2: Estimate g from the examples.\n'
        f'Step 3: The inferred g is approximately {g:.4f}.\n'
        f'Step 4: Substitute t = {target_t:.2f} into d = 0.5 * g * t^2.\n'
        f'Step 5: The computed distance is {formatted_pred}.\n'
        f'Final answer: \boxed{{{answer}}}'
    )
    return trace

def parse_unit_conversion_examples(prompt):
    examples = re.findall(r'([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z]+)\s+becomes\s+([0-9]+(?:\.[0-9]+)?)', prompt, flags=re.IGNORECASE)
    query = re.search(r'convert the following (?:measurement|value):\s*([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z]+)', prompt, flags=re.IGNORECASE)
    if not examples or not query:
        return None
    ex = [(float(x), unit, float(y)) for x, unit, y in examples]
    target_x = float(query.group(1))
    target_unit = query.group(2)
    return ex, target_x, target_unit

def build_unit_conversion_trace(row):
    prompt = row['prompt']
    answer = str(row['answer']).strip()
    parsed = parse_unit_conversion_examples(prompt)
    if parsed is None:
        return None
    ex, target_x, target_unit = parsed

    deltas = [y - x for x, _, y in ex]
    avg_delta = sum(deltas) / len(deltas)
    pred = target_x + avg_delta
    formatted_pred = format_answer_like_ground_truth(pred, answer)

    trace = (
        f'Step 1: Compare each example input and output in {target_unit}.\n'
        f'Step 2: Infer the hidden conversion rule from the examples.\n'
        f'Step 3: The average shift from the examples is {avg_delta:.4f}.\n'
        f'Step 4: Apply the same rule to {target_x:.2f} {target_unit}.\n'
        f'Step 5: The computed converted value is {formatted_pred}.\n'
        f'Final answer: \boxed{{{answer}}}'
    )
    return trace


In [ ]:
def build_trace(row):
    category = classify_problem(row['prompt'])
    if category == 'numeral':
        return category, build_numeral_trace(row)
    if category == 'gravity':
        return category, build_gravity_trace(row)
    if category == 'unit_conversion':
        return category, build_unit_conversion_trace(row)
    return category, None

records = []
skipped = []

for row in train.to_dicts():
    category, trace = build_trace(row)
    user_content = (
        row['prompt']
        + '\nPlease put your final answer inside \boxed{}. For example: \boxed{your answer}'
    )

    if trace is None:
        skipped.append({
            'id': row['id'],
            'category': category,
            'answer': row['answer'],
            'reason': 'unsupported_or_parse_failed',
        })
        continue

    rec = {
        'id': row['id'],
        'category': category,
        'answer': row['answer'],
        'messages': [
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': trace},
        ],
        'assistant_text': trace,
        'approx_trace_chars': len(trace),
    }
    records.append(rec)

print('Generated records:', len(records))
print('Skipped records:', len(skipped))


In [ ]:
with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

preview_rows = []
for rec in records:
    preview_rows.append({
        'id': rec['id'],
        'category': rec['category'],
        'answer': rec['answer'],
        'approx_trace_chars': rec['approx_trace_chars'],
        'assistant_preview': rec['assistant_text'][:300],
    })

pd.DataFrame(preview_rows).to_csv(PREVIEW_PATH, index=False)

stats_df = pd.DataFrame([
    {'category': k, 'count': v} for k, v in pd.Series([r['category'] for r in records]).value_counts().to_dict().items()
])
stats_df.to_csv(STATS_PATH, index=False)

pd.DataFrame(skipped).to_csv(SKIPPED_PATH, index=False)

print('Saved:', JSONL_PATH)
print('Saved:', PREVIEW_PATH)
print('Saved:', STATS_PATH)
print('Saved:', SKIPPED_PATH)


In [ ]:
print('Category counts:')
if len(records) > 0:
    print(pd.Series([r['category'] for r in records]).value_counts())

print('\nFirst examples by category:')
seen = set()
for rec in records:
    cat = rec['category']
    if cat in seen:
        continue
    seen.add(cat)
    print('\n===', cat, '===')
    print(rec['assistant_text'][:800])

print('\nDone.')
